# Benchmark: Basic Pitch vs Basic Pitch + DSP

Este notebook evalúa la fidelidad de la transcripción automática (Audio → MIDI) del modelo **Basic Pitch** (Spotify) 
comparando su salida cruda contra una versión procesada con filtros DSP personalizados.

**Dataset:** BabySlakh (stems aislados + MIDI Ground Truth alineados)  
**Instrumento evaluado:** Bass (S03)  
**Métricas (`mir_eval`):**
- $F_{no}$ — F-measure de Onset/Pitch (tolerancia de onset: 50ms)
- $F$ — F-measure de Onset/Pitch/Offset (offset: 20% duración, mín 50ms)
- $Acc$ — Accuracy a nivel de frame (multipitch, ventana 10ms)

---
## 1. Configuración y Carga de Datos

In [ ]:
import numpy as np
import librosa
import pretty_midi
import mir_eval
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import pandas as pd
import warnings

warnings.filterwarnings('ignore')

# ─── Constantes globales ────────────────────────────────────────────────────
TARGET_SR = 22050          # Sample rate nativo de Basic Pitch
ONSET_TOL = 0.05           # Tolerancia de onset: 50 ms
FRAME_HOP_SEC = 0.01       # Resolución de frame para multipitch: 10 ms
MIDI_PITCH_MIN = 24        # C1 — límite inferior del piano roll
MIDI_PITCH_MAX = 72        # C5 — límite superior del piano roll

print("✅ Librerías cargadas correctamente.")

In [ ]:
# ─── Rutas al track de ejemplo ──────────────────────────────────────────────
TRACK_DIR  = "data/babyslakh_16k/Track00001"
AUDIO_PATH = f"{TRACK_DIR}/stems/S03.wav"     # Stem aislado de Bass
MIDI_PATH  = f"{TRACK_DIR}/MIDI/S03.mid"      # Ground Truth MIDI

print(f"Audio: {AUDIO_PATH}")
print(f"MIDI:  {MIDI_PATH}")

In [ ]:
def load_audio(audio_path: str, target_sr: int = TARGET_SR) -> tuple[np.ndarray, int]:
    """
    Carga un archivo .wav y lo resamplea a la tasa objetivo.
    
    Args:
        audio_path: Ruta al archivo de audio.
        target_sr:  Sample rate deseado (default: 22050 Hz).
    
    Returns:
        Tupla (señal mono como np.ndarray, sample rate).
    """
    y, sr = librosa.load(audio_path, sr=target_sr, mono=True)
    print(f"  Audio cargado: {len(y)/sr:.2f}s | SR={sr} Hz | {len(y)} samples")
    return y, sr


def load_ground_truth_midi(midi_path: str) -> pretty_midi.PrettyMIDI:
    """
    Carga un archivo MIDI de Ground Truth.
    
    Args:
        midi_path: Ruta al archivo .mid.
    
    Returns:
        Objeto PrettyMIDI con las notas del Ground Truth.
    """
    pm = pretty_midi.PrettyMIDI(midi_path)
    total_notes = sum(len(inst.notes) for inst in pm.instruments)
    print(f"  MIDI cargado: {total_notes} notas | {pm.instruments[0].program if pm.instruments else '?'} program")
    return pm

In [ ]:
# ─── Cargar datos ───────────────────────────────────────────────────────────
print("Cargando audio...")
audio_signal, sr = load_audio(AUDIO_PATH)

print("\nCargando Ground Truth MIDI...")
gt_midi = load_ground_truth_midi(MIDI_PATH)

---
## 2. Inferencia con Basic Pitch

In [ ]:
from basic_pitch.inference import predict
from basic_pitch import ICASSP_2022_MODEL_PATH

def run_basic_pitch(audio_path: str) -> dict:
    """
    Ejecuta la inferencia de Basic Pitch sobre un archivo de audio.
    
    Args:
        audio_path: Ruta al archivo .wav.
    
    Returns:
        Diccionario con las siguientes claves:
        - 'midi':           Objeto PrettyMIDI con las notas detectadas.
        - 'note_events':    Lista de tuplas (onset, offset, pitch, velocity, [pitch_bends]).
        - 'contours':       Pitch contours a nivel de frame (np.ndarray).
        - 'onsets':         Activaciones de onset a nivel de frame (np.ndarray).
        - 'notes':          Activaciones de nota a nivel de frame (np.ndarray).
    """
    # predict() devuelve: (model_output, midi_data, note_events)
    # model_output es un dict con 'onset', 'contour', 'note'
    model_output, midi_data, note_events = predict(
        audio_path,
        model_or_model_path=ICASSP_2022_MODEL_PATH,
    )
    
    total_notes = sum(len(inst.notes) for inst in midi_data.instruments)
    print(f"  Basic Pitch detectó {total_notes} notas")
    print(f"  Contours shape: {model_output['contour'].shape}")
    
    return {
        'midi':        midi_data,
        'note_events': note_events,
        'contours':    model_output['contour'],
        'onsets':      model_output['onset'],
        'notes':       model_output['note'],
    }

In [ ]:
# ─── Ejecutar inferencia ────────────────────────────────────────────────────
print("Ejecutando Basic Pitch...")
bp_output = run_basic_pitch(AUDIO_PATH)
print("\n✅ Inferencia completada.")

---
## 3. Procesamiento DSP Personalizado

Esta función es un **placeholder** donde puedes inyectar tu lógica de filtrado DSP.
Recibe tanto el audio como las notas detectadas, y devuelve versiones procesadas de ambos.

Los ejemplos comentados muestran filtros típicos que puedes activar.

In [ ]:
def apply_custom_dsp(
    audio: np.ndarray,
    sr: int,
    note_events: list,
    bp_midi: pretty_midi.PrettyMIDI,
) -> tuple[np.ndarray, list, pretty_midi.PrettyMIDI]:
    """
    Placeholder para procesamiento DSP personalizado.
    
    Recibe el audio crudo y las notas detectadas por Basic Pitch.
    Devuelve versiones filtradas/limpias de ambos.
    
    Args:
        audio:       Señal de audio (np.ndarray, mono).
        sr:          Sample rate del audio.
        note_events: Lista de tuplas (onset, offset, pitch, velocity, [pitch_bends]).
        bp_midi:     Objeto PrettyMIDI con las notas de Basic Pitch.
    
    Returns:
        Tupla (audio_procesado, note_events_filtrados, midi_filtrado).
    """
    # ╔══════════════════════════════════════════════════════════════════════╗
    # ║  INYECTA TU LÓGICA DE FILTRADO AQUÍ                                ║
    # ║  Descomenta y ajusta los ejemplos según tu necesidad.               ║
    # ╚══════════════════════════════════════════════════════════════════════╝
    
    processed_audio = audio.copy()
    filtered_notes = list(note_events)  # copia de las notas
    
    # ──────────────────────────────────────────────────────────────────────
    # Ejemplo 1: Filtro de duración mínima (eliminar notas fantasma)
    # ──────────────────────────────────────────────────────────────────────
    MIN_DURATION_SEC = 0.05  # 50 ms
    filtered_notes = [
        note for note in filtered_notes
        if (note[1] - note[0]) >= MIN_DURATION_SEC
    ]
    
    # ──────────────────────────────────────────────────────────────────────
    # Ejemplo 2: Filtro de rango de pitch (mantener solo rango del bajo)
    # ──────────────────────────────────────────────────────────────────────
    PITCH_MIN, PITCH_MAX = 28, 67  # E1 a G4 (rango típico de bajo)
    filtered_notes = [
        note for note in filtered_notes
        if PITCH_MIN <= note[2] <= PITCH_MAX
    ]
    
    # ──────────────────────────────────────────────────────────────────────
    # Ejemplo 3: Filtro de velocidad mínima (eliminar notas muy débiles)
    # ──────────────────────────────────────────────────────────────────────
    # MIN_VELOCITY = 30
    filtered_notes = [
        note for note in filtered_notes
    #     if note[3] >= MIN_VELOCITY
    ]
    
    # ──────────────────────────────────────────────────────────────────────
    # Ejemplo 4: DSP en audio con Pedalboard (pre-procesamiento)
    # Si aplicas esto, deberás re-inferir con Basic Pitch después.
    # ──────────────────────────────────────────────────────────────────────
    # from pedalboard import Pedalboard, HighpassFilter, NoiseGate, Compressor
    # board = Pedalboard([
    #     HighpassFilter(cutoff_frequency_hz=30.0),   # Eliminar sub-graves
    #     NoiseGate(threshold_db=-40, ratio=10.0),    # Gate para silenciar residuos
    #     Compressor(threshold_db=-20, ratio=4.0),    # Estabilizar dinámica
    ]
    # processed_audio = board(audio, sr)
    
    # ──────────────────────────────────────────────────────────────────────
    # Reconstruir el PrettyMIDI a partir de las notas filtradas
    # ──────────────────────────────────────────────────────────────────────
    filtered_midi = pretty_midi.PrettyMIDI()
    instrument = pretty_midi.Instrument(program=33, name='Electric Bass (finger)')  # GM program para bajo
    for note_event in filtered_notes:
        onset, offset, pitch, velocity = note_event[0], note_event[1], note_event[2], note_event[3]
        note = pretty_midi.Note(
            velocity=int(velocity),
            pitch=int(pitch),
            start=onset,
            end=offset,
        )
        instrument.notes.append(note)
    filtered_midi.instruments.append(instrument)
    
    total = sum(len(inst.notes) for inst in filtered_midi.instruments)
    print(f"  DSP: {len(note_events)} notas → {total} notas (filtradas {len(note_events) - total})")
    
    return processed_audio, filtered_notes, filtered_midi

In [ ]:
# ─── Aplicar DSP personalizado ──────────────────────────────────────────────
print("Aplicando DSP personalizado...")
processed_audio, dsp_note_events, dsp_midi = apply_custom_dsp(
    audio=audio_signal,
    sr=sr,
    note_events=bp_output['note_events'],
    bp_midi=bp_output['midi'],
)
print("✅ DSP aplicado.")

---
## 4. Cálculo de Métricas (`mir_eval`)

### 4.1 Funciones auxiliares de conversión

In [ ]:
def midi_to_intervals_and_pitches(pm: pretty_midi.PrettyMIDI) -> tuple[np.ndarray, np.ndarray]:
    """
    Extrae intervalos (onset, offset) y frecuencias en Hz de un objeto PrettyMIDI.
    
    mir_eval.transcription requiere:
    - intervals: np.ndarray de shape (n, 2) con [onset, offset] en segundos.
    - pitches:   np.ndarray de shape (n,) con frecuencias en Hz.
    
    Returns:
        Tupla (intervals, pitches_hz).
    """
    all_notes = []
    for inst in pm.instruments:
        if not inst.is_drum:
            for note in inst.notes:
                all_notes.append((note.start, note.end, note.pitch))
    
    if not all_notes:
        return np.zeros((0, 2)), np.zeros(0)
    
    # Ordenar por onset
    all_notes.sort(key=lambda x: x[0])
    
    intervals = np.array([[n[0], n[1]] for n in all_notes])
    pitches_hz = np.array([pretty_midi.note_number_to_hz(n[2]) for n in all_notes])
    
    return intervals, pitches_hz


def midi_to_frame_frequencies(
    pm: pretty_midi.PrettyMIDI,
    total_duration: float,
    hop_sec: float = FRAME_HOP_SEC,
) -> list[np.ndarray]:
    """
    Convierte notas MIDI a una lista de frecuencias activas por frame.
    
    mir_eval.multipitch.metrics requiere:
    - Una lista de arrays, donde cada array contiene las frecuencias
      activas (en Hz) en ese frame temporal.
    
    Args:
        pm:             Objeto PrettyMIDI.
        total_duration: Duración total en segundos.
        hop_sec:        Resolución temporal por frame (default: 10 ms).
    
    Returns:
        Lista de np.ndarray con frecuencias activas por frame.
    """
    n_frames = int(np.ceil(total_duration / hop_sec))
    frame_times = np.arange(n_frames) * hop_sec
    
    # Recopilar todas las notas (excluyendo drums)
    all_notes = []
    for inst in pm.instruments:
        if not inst.is_drum:
            for note in inst.notes:
                all_notes.append((note.start, note.end, note.pitch))
    
    # Para cada frame, encontrar las frecuencias activas
    frame_freqs = []
    for t in frame_times:
        active_freqs = []
        for start, end, pitch in all_notes:
            if start <= t < end:
                active_freqs.append(pretty_midi.note_number_to_hz(pitch))
        frame_freqs.append(np.array(active_freqs))
    
    return frame_freqs

### 4.2 Cálculo de $F_{no}$ y $F$ (transcripción nota a nota)

In [ ]:
def compute_transcription_metrics(
    gt_midi: pretty_midi.PrettyMIDI,
    pred_midi: pretty_midi.PrettyMIDI,
    onset_tolerance: float = ONSET_TOL,
) -> dict:
    """
    Calcula las métricas de transcripción nota-a-nota con mir_eval.
    
    Métricas calculadas:
    - F_no:  F-measure considerando solo Onset + Pitch (sin offset).
    - F:     F-measure considerando Onset + Pitch + Offset.
    
    Args:
        gt_midi:          PrettyMIDI del Ground Truth.
        pred_midi:        PrettyMIDI de la predicción.
        onset_tolerance:  Tolerancia de onset en segundos (default: 0.05).
    
    Returns:
        Diccionario con P_no, R_no, F_no, P, R, F.
    """
    # Extraer intervalos y frecuencias
    ref_intervals, ref_pitches = midi_to_intervals_and_pitches(gt_midi)
    est_intervals, est_pitches = midi_to_intervals_and_pitches(pred_midi)
    
    # ── F_no: Onset + Pitch (sin offset) ────────────────────────────────
    # offset_ratio=None desactiva la evaluación de offset
    P_no, R_no, F_no, _ = mir_eval.transcription.precision_recall_f1_overlap(
        ref_intervals, ref_pitches,
        est_intervals, est_pitches,
        onset_tolerance=onset_tolerance,
        offset_ratio=None,  # ← desactivar offset
    )
    
    # ── F: Onset + Pitch + Offset ────────────────────────────────────────
    # offset_ratio=0.2 → tolerancia = max(50ms, 20% de la duración de la nota)
    P, R, F, _ = mir_eval.transcription.precision_recall_f1_overlap(
        ref_intervals, ref_pitches,
        est_intervals, est_pitches,
        onset_tolerance=onset_tolerance,
        offset_ratio=0.2,
        offset_min_tolerance=0.05,
    )
    
    return {
        'P_no': P_no, 'R_no': R_no, 'F_no': F_no,
        'P': P, 'R': R, 'F': F,
    }

### 4.3 Cálculo de $Acc$ (Accuracy a nivel de frame)

In [ ]:
def compute_frame_accuracy(
    gt_midi: pretty_midi.PrettyMIDI,
    pred_midi: pretty_midi.PrettyMIDI,
    total_duration: float,
    hop_sec: float = FRAME_HOP_SEC,
) -> float:
    """
    Calcula la Accuracy a nivel de frame usando mir_eval.multipitch.
    
    Genera las frecuencias fundamentales activas por frame (cada 10ms)
    tanto para el Ground Truth como para la predicción, y calcula
    la métrica 'Accuracy' de mir_eval.multipitch.metrics.
    
    Args:
        gt_midi:        PrettyMIDI del Ground Truth.
        pred_midi:      PrettyMIDI de la predicción.
        total_duration: Duración total del audio en segundos.
        hop_sec:        Resolución temporal por frame.
    
    Returns:
        Accuracy (float entre 0 y 1).
    """
    # Generar tiempos de frame
    n_frames = int(np.ceil(total_duration / hop_sec))
    frame_times = np.arange(n_frames) * hop_sec
    
    # Extraer frecuencias por frame
    ref_freqs = midi_to_frame_frequencies(gt_midi, total_duration, hop_sec)
    est_freqs = midi_to_frame_frequencies(pred_midi, total_duration, hop_sec)
    
    # Asegurar misma longitud
    min_len = min(len(ref_freqs), len(est_freqs), len(frame_times))
    ref_freqs = ref_freqs[:min_len]
    est_freqs = est_freqs[:min_len]
    frame_times = frame_times[:min_len]
    
    # mir_eval.multipitch.metrics() devuelve una TUPLA de 14 valores:
    # (precision, recall, accuracy, e_sub, e_miss, e_fa, e_tot,
    #  precision_chroma, recall_chroma, accuracy_chroma,
    #  e_sub_chroma, e_miss_chroma, e_fa_chroma, e_tot_chroma)
    result = mir_eval.multipitch.metrics(
        frame_times, ref_freqs,
        frame_times, est_freqs,
    )
    
    # Accuracy es el tercer elemento (índice 2)
    precision, recall, accuracy = result[0], result[1], result[2]
    return accuracy

### 4.4 Calcular métricas para ambas condiciones

In [ ]:
total_duration = len(audio_signal) / sr

# ── Métricas: Basic Pitch Crudo ──────────────────────────────────────────
print("Calculando métricas para Basic Pitch Crudo...")
bp_trans_metrics = compute_transcription_metrics(gt_midi, bp_output['midi'])
bp_frame_acc = compute_frame_accuracy(gt_midi, bp_output['midi'], total_duration)

print(f"  F_no = {bp_trans_metrics['F_no']:.4f}")
print(f"  F    = {bp_trans_metrics['F']:.4f}")
print(f"  Acc  = {bp_frame_acc:.4f}")

# ── Métricas: Basic Pitch + DSP ──────────────────────────────────────────
print("\nCalculando métricas para Basic Pitch + DSP...")
dsp_trans_metrics = compute_transcription_metrics(gt_midi, dsp_midi)
dsp_frame_acc = compute_frame_accuracy(gt_midi, dsp_midi, total_duration)

print(f"  F_no = {dsp_trans_metrics['F_no']:.4f}")
print(f"  F    = {dsp_trans_metrics['F']:.4f}")
print(f"  Acc  = {dsp_frame_acc:.4f}")

---
## 5. Visualización Comparativa

### 5.1 Reproducción de Audio

Escucha comparativa de las tres señales:
1. **Audio Original** — el stem aislado de BabySlakh.
2. **Basic Pitch (sintetizado)** — las notas detectadas por BP, resintentizadas a audio.
3. **Basic Pitch + DSP (sintetizado)** — las notas filtradas por tu DSP, resintetizadas.

In [ ]:
import IPython.display as ipd

# ─── Sintetizar MIDI a audio para escucha comparativa ───────────────────────
# pretty_midi.synthesize() usa síntesis aditiva (ondas sinusoidales).
# No suena como un bajo real, pero permite comparar las notas detectadas.

def synthesize_midi(pm: pretty_midi.PrettyMIDI, fs: int = TARGET_SR) -> np.ndarray:
    """
    Sintetiza un objeto PrettyMIDI a una señal de audio.
    Normaliza para evitar clipping.
    """
    audio = pm.synthesize(fs=fs)
    if np.max(np.abs(audio)) > 0:
        audio = audio / np.max(np.abs(audio)) * 0.9
    return audio

bp_synth = synthesize_midi(bp_output["midi"])
dsp_synth = synthesize_midi(dsp_midi)

print("🎵 Audio Original (stem aislado):")
ipd.display(ipd.Audio(audio_signal, rate=sr))

print("
🤖 Basic Pitch — notas detectadas (sintetizado):")
ipd.display(ipd.Audio(bp_synth, rate=sr))

print("
🔧 Basic Pitch + DSP — notas filtradas (sintetizado):")
ipd.display(ipd.Audio(dsp_synth, rate=sr))

In [ ]:
# ─── Bonus: sintetizar el Ground Truth MIDI para referencia ─────────────────
gt_synth = synthesize_midi(gt_midi)

print("📋 Ground Truth MIDI (sintetizado):")
ipd.display(ipd.Audio(gt_synth, rate=sr))

In [ ]:
def classify_notes(
    gt_midi: pretty_midi.PrettyMIDI,
    pred_midi: pretty_midi.PrettyMIDI,
    onset_tolerance: float = ONSET_TOL,
    pitch_tolerance: float = 50.0,  # cents
) -> tuple[list, list, list]:
    """
    Clasifica las notas predichas en True Positives, False Positives
    y las notas GT no detectadas como False Negatives.
    
    Un TP se define como una nota predicha que tiene:
    - onset dentro de `onset_tolerance` de una nota GT
    - mismo pitch MIDI
    
    Returns:
        (true_positives, false_positives, false_negatives)
        Cada uno es una lista de tuplas (onset, offset, pitch).
    """
    # Recopilar notas
    gt_notes = []
    for inst in gt_midi.instruments:
        if not inst.is_drum:
            for n in inst.notes:
                gt_notes.append((n.start, n.end, n.pitch))
    
    pred_notes = []
    for inst in pred_midi.instruments:
        if not inst.is_drum:
            for n in inst.notes:
                pred_notes.append((n.start, n.end, n.pitch))
    
    gt_notes.sort(key=lambda x: x[0])
    pred_notes.sort(key=lambda x: x[0])
    
    # Matching greedy: para cada nota predicha, buscar match en GT
    gt_matched = set()
    tp, fp = [], []
    
    for pred in pred_notes:
        matched = False
        for i, gt in enumerate(gt_notes):
            if i in gt_matched:
                continue
            if (abs(pred[0] - gt[0]) <= onset_tolerance and pred[2] == gt[2]):
                tp.append(pred)
                gt_matched.add(i)
                matched = True
                break
        if not matched:
            fp.append(pred)
    
    # False Negatives: notas GT sin match
    fn = [gt_notes[i] for i in range(len(gt_notes)) if i not in gt_matched]
    
    return tp, fp, fn


def plot_piano_roll(
    gt_midi: pretty_midi.PrettyMIDI,
    pred_midi: pretty_midi.PrettyMIDI,
    title: str = "Piano Roll: Ground Truth vs Predicción",
    pitch_min: int = MIDI_PITCH_MIN,
    pitch_max: int = MIDI_PITCH_MAX,
):
    """
    Dibuja un piano roll superpuesto con colores para TP, FP y FN.
    
    - Verde:     True Positives  (nota correctamente detectada)
    - Rojo:      False Positives (nota fantasma / falso positivo)
    - Azul/Gris: False Negatives (nota GT no detectada)
    """
    tp, fp, fn = classify_notes(gt_midi, pred_midi)
    
    fig, ax = plt.subplots(figsize=(18, 6))
    
    bar_height = 0.7
    
    # Dibujar False Negatives (fondo, azul/gris)
    for onset, offset, pitch in fn:
        if pitch_min <= pitch <= pitch_max:
            ax.barh(
                pitch, offset - onset, left=onset,
                height=bar_height, color='#5B7FA3', alpha=0.5,
                edgecolor='#3D5A80', linewidth=0.5,
            )
    
    # Dibujar False Positives (rojo)
    for onset, offset, pitch in fp:
        if pitch_min <= pitch <= pitch_max:
            ax.barh(
                pitch, offset - onset, left=onset,
                height=bar_height, color='#E74C3C', alpha=0.7,
                edgecolor='#C0392B', linewidth=0.5,
            )
    
    # Dibujar True Positives (verde, al frente)
    for onset, offset, pitch in tp:
        if pitch_min <= pitch <= pitch_max:
            ax.barh(
                pitch, offset - onset, left=onset,
                height=bar_height, color='#2ECC71', alpha=0.85,
                edgecolor='#27AE60', linewidth=0.5,
            )
    
    # Configuración de ejes
    ax.set_ylim(pitch_min - 1, pitch_max + 1)
    ax.set_ylabel('Nota MIDI')
    ax.set_xlabel('Tiempo (s)')
    ax.set_title(title, fontsize=14, fontweight='bold')
    
    # Etiquetas de notas MIDI en eje Y (cada 2 semitonos)
    yticks = list(range(pitch_min, pitch_max + 1, 2))
    ax.set_yticks(yticks)
    ax.set_yticklabels(
        [pretty_midi.note_number_to_name(p) for p in yticks],
        fontsize=7,
    )
    
    # Leyenda
    legend_handles = [
        mpatches.Patch(color='#2ECC71', alpha=0.85, label=f'True Positive ({len(tp)})'),
        mpatches.Patch(color='#E74C3C', alpha=0.7,  label=f'False Positive ({len(fp)})'),
        mpatches.Patch(color='#5B7FA3', alpha=0.5,  label=f'False Negative ({len(fn)})'),
    ]
    ax.legend(handles=legend_handles, loc='upper right', fontsize=9)
    
    ax.grid(axis='y', alpha=0.2)
    plt.tight_layout()
    plt.show()
    
    print(f"  TP: {len(tp)} | FP: {len(fp)} | FN: {len(fn)}")

In [ ]:
# ─── Piano Roll: Basic Pitch Crudo ──────────────────────────────────────────
plot_piano_roll(
    gt_midi, bp_output['midi'],
    title="Piano Roll — Basic Pitch Crudo vs Ground Truth",
)

In [ ]:
# ─── Piano Roll: Basic Pitch + DSP ──────────────────────────────────────────
plot_piano_roll(
    gt_midi, dsp_midi,
    title="Piano Roll — Basic Pitch + DSP vs Ground Truth",
)

### 5.3 Matrices de Confusión

Se presentan dos tipos de matrices de confusión:
1. **A nivel de frame (2×2):** Por cada frame de 10ms, se evalúa si hay actividad de nota (Activo/Inactivo) tanto en el Ground Truth como en la predicción.
2. **Desglose por pitch:** Para cada nota MIDI presente, cuántas fueron TP, FP o FN.

In [ ]:
def compute_frame_confusion_matrix(
    gt_midi: pretty_midi.PrettyMIDI,
    pred_midi: pretty_midi.PrettyMIDI,
    total_duration: float,
    hop_sec: float = FRAME_HOP_SEC,
) -> np.ndarray:
    """
    Calcula una matriz de confusión 2x2 a nivel de frame.
    
    Cada frame (10ms) se clasifica como:
    - True Negative  (TN): ni GT ni predicción tienen notas activas.
    - False Positive (FP): solo la predicción tiene notas activas.
    - False Negative (FN): solo el GT tiene notas activas.
    - True Positive  (TP): ambos tienen notas activas.
    
    Returns:
        np.ndarray de shape (2, 2): [[TN, FP], [FN, TP]]
    """
    ref_freqs = midi_to_frame_frequencies(gt_midi, total_duration, hop_sec)
    est_freqs = midi_to_frame_frequencies(pred_midi, total_duration, hop_sec)
    
    min_len = min(len(ref_freqs), len(est_freqs))
    
    tn = fp = fn = tp = 0
    for i in range(min_len):
        ref_active = len(ref_freqs[i]) > 0
        est_active = len(est_freqs[i]) > 0
        
        if not ref_active and not est_active:
            tn += 1
        elif not ref_active and est_active:
            fp += 1
        elif ref_active and not est_active:
            fn += 1
        else:  # ambos activos
            tp += 1
    
    return np.array([[tn, fp], [fn, tp]])


def plot_confusion_matrices(
    gt_midi: pretty_midi.PrettyMIDI,
    bp_midi: pretty_midi.PrettyMIDI,
    dsp_midi: pretty_midi.PrettyMIDI,
    total_duration: float,
):
    """
    Dibuja lado a lado las matrices de confusión (frame-level)
    para Basic Pitch Crudo y Basic Pitch + DSP.
    """
    cm_bp = compute_frame_confusion_matrix(gt_midi, bp_midi, total_duration)
    cm_dsp = compute_frame_confusion_matrix(gt_midi, dsp_midi, total_duration)
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    labels = ['Inactivo', 'Activo']
    
    for ax, cm, title in zip(
        axes,
        [cm_bp, cm_dsp],
        ['Basic Pitch Crudo', 'Basic Pitch + DSP'],
    ):
        # Calcular porcentajes
        total_frames = cm.sum()
        cm_pct = cm / total_frames * 100
        
        # Crear anotaciones con conteo y porcentaje
        annot = np.empty_like(cm, dtype=object)
        for i in range(2):
            for j in range(2):
                annot[i, j] = f'{cm[i, j]:,}\n({cm_pct[i, j]:.1f}%)'
        
        # Colores: verde para diagonal (TN, TP), rojo para off-diagonal (FP, FN)
        colors = np.array([
            ['#d4edda', '#f8d7da'],  # TN (verde claro), FP (rojo claro)
            ['#d6e9f8', '#c3e6cb'],  # FN (azul claro),  TP (verde)
        ])
        
        # Dibujar heatmap manual
        ax.imshow(cm_pct, cmap='Greens', alpha=0.6, aspect='auto', vmin=0)
        
        for i in range(2):
            for j in range(2):
                ax.text(j, i, annot[i, j],
                        ha='center', va='center', fontsize=13, fontweight='bold')
        
        ax.set_xticks([0, 1])
        ax.set_yticks([0, 1])
        ax.set_xticklabels(labels, fontsize=11)
        ax.set_yticklabels(labels, fontsize=11)
        ax.set_xlabel('Predicción', fontsize=12)
        ax.set_ylabel('Ground Truth', fontsize=12)
        ax.set_title(f'Matriz de Confusión (Frame)\n{title}', fontsize=13, fontweight='bold')
    
    plt.tight_layout()
    plt.show()

In [ ]:
# ─── Matrices de confusión a nivel de frame ─────────────────────────────────
plot_confusion_matrices(gt_midi, bp_output['midi'], dsp_midi, total_duration)

In [ ]:
def plot_pitch_breakdown(
    gt_midi: pretty_midi.PrettyMIDI,
    pred_midi: pretty_midi.PrettyMIDI,
    title: str = "Desglose TP / FP / FN por Pitch",
):
    """
    Gráfico de barras apiladas mostrando, para cada nota MIDI
    que aparece en GT o predicción, cuántas fueron TP, FP o FN.
    """
    tp, fp, fn = classify_notes(gt_midi, pred_midi)
    
    # Contar por pitch
    from collections import Counter
    tp_counts = Counter(n[2] for n in tp)
    fp_counts = Counter(n[2] for n in fp)
    fn_counts = Counter(n[2] for n in fn)
    
    # Todos los pitches presentes
    all_pitches = sorted(set(tp_counts) | set(fp_counts) | set(fn_counts))
    
    if not all_pitches:
        print("  No hay notas para graficar.")
        return
    
    tp_vals = [tp_counts.get(p, 0) for p in all_pitches]
    fp_vals = [fp_counts.get(p, 0) for p in all_pitches]
    fn_vals = [fn_counts.get(p, 0) for p in all_pitches]
    
    pitch_labels = [pretty_midi.note_number_to_name(p) for p in all_pitches]
    x = np.arange(len(all_pitches))
    bar_w = 0.6
    
    fig, ax = plt.subplots(figsize=(max(10, len(all_pitches) * 0.6), 5))
    
    ax.bar(x, tp_vals, bar_w, label='True Positive', color='#2ECC71', alpha=0.85)
    ax.bar(x, fp_vals, bar_w, bottom=tp_vals, label='False Positive', color='#E74C3C', alpha=0.7)
    ax.bar(x, fn_vals, bar_w,
           bottom=[t + f for t, f in zip(tp_vals, fp_vals)],
           label='False Negative', color='#5B7FA3', alpha=0.5)
    
    ax.set_xticks(x)
    ax.set_xticklabels(pitch_labels, rotation=45, ha='right', fontsize=8)
    ax.set_xlabel('Nota MIDI')
    ax.set_ylabel('Cantidad de notas')
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.legend(fontsize=9)
    ax.grid(axis='y', alpha=0.3)
    
    plt.tight_layout()
    plt.show()

In [ ]:
# ─── Desglose por pitch: Basic Pitch Crudo ──────────────────────────────────
plot_pitch_breakdown(
    gt_midi, bp_output['midi'],
    title="Desglose TP / FP / FN por Pitch — Basic Pitch Crudo",
)

In [ ]:
# ─── Desglose por pitch: Basic Pitch + DSP ─────────────────────────────────
plot_pitch_breakdown(
    gt_midi, dsp_midi,
    title="Desglose TP / FP / FN por Pitch — Basic Pitch + DSP",
)

### 5.4 Tabla de Benchmarking

In [ ]:
# ─── Construir DataFrame comparativo ────────────────────────────────────────
benchmark_data = {
    'Condición': ['Basic Pitch Crudo', 'Basic Pitch + DSP'],
    'Acc (Frame)': [
        round(bp_frame_acc, 4),
        round(dsp_frame_acc, 4),
    ],
    'F_no (Onset)': [
        round(bp_trans_metrics['F_no'], 4),
        round(dsp_trans_metrics['F_no'], 4),
    ],
    'F (Onset+Offset)': [
        round(bp_trans_metrics['F'], 4),
        round(dsp_trans_metrics['F'], 4),
    ],
}

df = pd.DataFrame(benchmark_data).set_index('Condición')

# Aplicar gradiente de color verde para resaltar mejores puntajes
styled_df = df.style.background_gradient(
    cmap='Greens',
    axis=0,        # Comparar dentro de cada columna
    vmin=0.0,
    vmax=1.0,
).format('{:.4f}').set_caption(
    'Benchmark: Basic Pitch Crudo vs Basic Pitch + DSP — Bass (Track00001)'
).set_table_styles([
    {'selector': 'caption', 'props': [('font-size', '14px'), ('font-weight', 'bold')]},
])

styled_df

In [ ]:
# ─── Detalle adicional: Precision y Recall ──────────────────────────────────
detail_data = {
    'Condición': ['Basic Pitch Crudo', 'Basic Pitch + DSP'],
    'P_no': [round(bp_trans_metrics['P_no'], 4), round(dsp_trans_metrics['P_no'], 4)],
    'R_no': [round(bp_trans_metrics['R_no'], 4), round(dsp_trans_metrics['R_no'], 4)],
    'F_no': [round(bp_trans_metrics['F_no'], 4), round(dsp_trans_metrics['F_no'], 4)],
    'P':    [round(bp_trans_metrics['P'], 4),    round(dsp_trans_metrics['P'], 4)],
    'R':    [round(bp_trans_metrics['R'], 4),    round(dsp_trans_metrics['R'], 4)],
    'F':    [round(bp_trans_metrics['F'], 4),    round(dsp_trans_metrics['F'], 4)],
    'Acc':  [round(bp_frame_acc, 4),             round(dsp_frame_acc, 4)],
}

df_detail = pd.DataFrame(detail_data).set_index('Condición')

df_detail.style.background_gradient(
    cmap='Greens', axis=0, vmin=0.0, vmax=1.0
).format('{:.4f}').set_caption(
    'Detalle: Precision, Recall y Accuracy por condición'
).set_table_styles([
    {'selector': 'caption', 'props': [('font-size', '14px'), ('font-weight', 'bold')]},
])